# Part 3: DIY Linear Regression

In this part, you will complete a Do-It-Yourself (DIY) implementation of ordinary least squares (OLS) linear regression in an object-oriented pattern that corresponds with the Scikit-Learn API.

**Learning objectives.** You will:
1. Write object-oriented code for a Python class, matching standard API patterns.
2. Apply numerical Python (NumPy) to efficiently implement ordinary least squares linear regression using the closed-form solution (normal equation).
3. Evaluate your implementation compared to the Scikit-Learn standard and simple baseline models on synthetic data.
4. Explore the effects of polynomial feature expansion on model performance and overfitting.

## Background: Linear Regression Theory

Linear regression models the relationship between a dependent variable $y$ and independent variables $\mathbf{x}$ as:

$$y = \mathbf{x}^T \mathbf{w} + \epsilon$$

where $\mathbf{w}$ are the model weights and $\epsilon$ is noise.

**Normal Equation (Closed-Form Solution):**
The maximum likelihood estimate (MLE) for the weights can be found analytically:

$$\mathbf{w}^* = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

where $\mathbf{X}$ is the design matrix with each row being a data point.

For numerical stability, it's better to solve the linear system $\mathbf{X}^T \mathbf{X} \mathbf{w} = \mathbf{X}^T \mathbf{y}$ directly rather than computing the matrix inverse.

## Task 1

First, we will establish baseline models to which we can compare our DIY implementation. Run the following code to generate synthetic data for use in this part of the assignment. Observe that the predictive target is continuous, and that the code also splits the synthetic data into train and test sets for you.

Implement and evaluate two baseline approaches:

1. **Constant baseline**: Predict the mean of the training targets for all examples (this is the optimal constant prediction under MSE loss).
2. **Scikit-Learn baseline**: Use Scikit-Learn to fit a [linear regression model](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) on the train set with default parameters.

For both baselines, evaluate and report the [mean squared error (MSE)](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html) on both the train set and the test set.

In [1]:
# Run but do not modify this code

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

np.random.seed(2025)
n = 100
features = 10

# Generate synthetic data
X = np.random.normal(size=(n, features))
true_weights = np.random.normal(size=features)
noise = np.random.normal(scale=0.5, size=n)
y = X @ true_weights + noise

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=2025)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Target statistics - Mean: {y_train.mean():.3f}, Std: {y_train.std():.3f}")

Training set size: (70, 10)
Test set size: (30, 10)
Target statistics - Mean: 0.241, Std: 3.236


In [2]:
# Write code for task 1 here
from sklearn.linear_model import LinearRegression as SklearnLinearRegression
from sklearn.metrics import mean_squared_error

# Mean of training target for baseline
y_mean = np.mean(y_train)
pred_const_train = np.full_like(y_train, fill_value=y_mean)
pred_const_test = np.full_like(y_test, fill_value=y_mean)

# Calculate train and test MSE
mse_const_train = mean_squared_error(y_train, pred_const_train)
mse_const_test = mean_squared_error(y_test, pred_const_test)

sk_model = SklearnLinearRegression()
sk_model.fit(X_train, y_train)

pred_sk_train = sk_model.predict(X_train)
pred_sk_test = sk_model.predict(X_test)

mse_sk_train = mean_squared_error(y_train, pred_sk_train)
mse_sk_test = mean_squared_error(y_test, pred_sk_test)

print(f"Constant Baseline MSE:\n Training Set: {mse_const_train} \n Test Set: {mse_const_test}")
print(f"SkLearn Baseline MSE:\n Training Set: {mse_sk_train} \n Test Set: {mse_sk_test}")


Constant Baseline MSE:
 Training Set: 10.470370767313392 
 Test Set: 10.471856783310526
SkLearn Baseline MSE:
 Training Set: 0.23159737368296543 
 Test Set: 0.2722365649138699


## Task 2

Complete the following class to implement linear regression using the closed-form solution. Some important notes about the implementation:

1. The Scikit-Learn API treats an input `X` array as a matrix with a row for every data point and a column for every feature.
2. For `fit`, every row in `X` corresponds to a given output in `y`. You don't need to return anything, just compute the optimal weights (stored as instance variables).
3. For `predict`, you should return a NumPy array with one predicted value for every row in the input `X`.
4. The number of weights in your model should equal the number of features (columns) in the `X` matrix.
5. **Closed-form solution**: Use the normal equation $\mathbf{w} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$. Use NumPy's [linalg.solve](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html) instead of computing the inverse directly for better numerical stability: solve $\mathbf{X}^T \mathbf{X} \mathbf{w} = \mathbf{X}^T \mathbf{y}$.
6. For predictions, use the dot product of the input features and learned weights.
7. Include debugging information when `verbose=True` in the `fit` method (e.g., print the computed weights).
8. Use vectorized NumPy operations to avoid slow for loops over large amounts of data.
9. You do **not** need to include a bias/intercept term for this implementation.

In [3]:
class LinearRegression:
    def __init__(self, random_state=2025):
        """
        Parameters:
        -----------
        random_state : int
            Random seed for reproducibility (not used in closed-form solution)
        """
        self.random_state = random_state
        self.weights = None

    def predict(self, X):
        """Predict continuous values for each row in X"""
        X = np.asarray(X)

        return X @ self.weights

    def fit(self, X, y, verbose=False):
        """Fit the training data using the normal equation.
        Parameters
        ----------
        X : {array-like}, shape = [n_examples, n_features]
          Training vectors, where n_examples is the number of examples and
          n_features is the number of features.
        y : array-like, shape = [n_examples]
          Target values (continuous).
        verbose : bool
          Whether to print training progress information.
        """
        X = np.asarray(X)
        y = np.asarray(y)

        # Form the matrix
        A = X.T @ X

        # Form the vector
        b = X.T @ y

        # Closed-form solution
        self.weights = np.linalg.solve(A, b)

        if verbose:
            print("Fit completed.")
            print(f"Condition number of X^T X: {np.linalg.cond(A)}")
            print(f"Learned weights (shape {self.weights.shape}):\n{self.weights}")

## Task 3

Use your DIY `LinearRegression` class from task 2 to fit a linear regression model on the train set. Evaluate and report the [mean squared error (MSE)](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html) on both the train set and the test set.

Compare the performance of your implementation to both baselines from Task 1 (constant model and Scikit-Learn). Your DIY implementation should achieve essentially identical performance to the Scikit-Learn implementation.

 **Note:** Due to numerical precision differences in floating-point arithmetic, small deviations in MSE values (less than 0.00001) can be considered as " identical".

In [4]:
# Write code for task 3 here
# Create and fit DIY model
diy_model = LinearRegression()
diy_model.fit(X_train, y_train)

# Predict on train and test sets
pred_diy_train = diy_model.predict(X_train)
pred_diy_test = diy_model.predict(X_test)

# Calculate MSE
mse_diy_train = mean_squared_error(y_train, pred_diy_train)
mse_diy_test = mean_squared_error(y_test, pred_diy_test)

#Output
print(f"DIY Model MSE:\n Training Set: {mse_diy_train}\n Test Set: {mse_diy_test}")
print(f"SkLearn Model MSE:\n Training Set: {mse_sk_train}\n Test Set: {mse_sk_test}")
print(f"Constant Baseline MSE:\n Training Set: {mse_const_train}\n Test Set: {mse_const_test}")

# Numerical equivalence checks
print("\nEquivalence Checks (DIY vs. SkLearn):")
print(f"  Training set difference: {abs(mse_diy_train - mse_sk_train)}")
print(f"  Are Training sets identical (Is the difference < 0.00001)? {abs(mse_diy_train - mse_sk_train) < 0.00001}")
print(f"  Test Set difference:  {abs(mse_diy_test - mse_sk_test)}")
print(f"  Are Test Sets identical (Is the difference < 0.00001)? {abs(mse_diy_test - mse_sk_test) < 0.00001}")

DIY Model MSE:
 Training Set: 0.2315973737193625
 Test Set: 0.27223833995486485
SkLearn Model MSE:
 Training Set: 0.23159737368296543
 Test Set: 0.2722365649138699
Constant Baseline MSE:
 Training Set: 10.470370767313392
 Test Set: 10.471856783310526

Equivalence Checks (DIY vs. SkLearn):
  Training set difference: 3.6397079794525666e-11
  Are Training sets identical (Is the difference < 0.00001)? True
  Test Set difference:  1.7750409949668366e-06
  Are Test Sets identical (Is the difference < 0.00001)? True


## Task 4

Now we will explore **polynomial feature expansion** to train a quadratic model and demonstrate overfitting. Create quadratic features from the original training data by computing all pairwise products of features. Specifically:

1. Generate polynomial features: For the original features $x_1, x_2, \ldots, x_d$, create all quadratic terms $x_i \cdot x_j$ for $i \leq j$. This includes squared terms like $x_1^2, x_2^2, \ldots$ and cross-product terms like $x_1 \cdot x_2, x_1 \cdot x_3, \ldots$. You can use Scikit-Learn's [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) with `degree=2` and `include_bias=False` to generate these features automatically or you can write your own NumPy code to generate these.

2. Use your DIY `LinearRegression` class to fit a model on the expanded feature set.

3. Evaluate and report the MSE on both train and test sets for the polynomial model. Compare these results to the linear model from Task 3.

4. Discuss what you observe about the training vs. test performance. Since the true underlying relationship is linear, what happens when you add quadratic features? Do you observe overfitting, underfitting, or neither?

In [19]:
# Write code for task 4 here
from sklearn.preprocessing import PolynomialFeatures

# Generate features (degree=2, no bias)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# Fit DIY model on polynomial
diy_poly_model = LinearRegression()
diy_poly_model.fit(X_train_poly, y_train)

# Predict and evaluate MSE on two sets
pred_poly_train = diy_poly_model.predict(X_train_poly)
pred_poly_test = diy_poly_model.predict(X_test_poly)

mse_poly_train = mean_squared_error(y_train, pred_poly_train)
mse_poly_test = mean_squared_error(y_test, pred_poly_test)

print(f"Number of original features: {X_train.shape[1]}")
print(f" Linear Model Training Set MSE: {mse_diy_train}")
print(f" Linear Model Test Set MSE: {mse_diy_test}\n")

print(f"Number of polynomial features: {X_train_poly.shape[1]}")
print(f" Polynomial Model Training Set MSE: {mse_poly_train}")
print(f" Polynomial Model Test Set MSE: {mse_poly_test}")

Number of original features: 10
 Linear Model Training Set MSE: 0.2315973737193625
 Linear Model Test Set MSE: 0.27223833995486485

Number of polynomial features: 65
 Polynomial Model Training Set MSE: 0.010110053760697789
 Polynomial Model Test Set MSE: 4.290069404611239


*Report and discuss findings about overfitting with polynomial features for task 4 here*

An observed disparity is present when viewing both the training and test MSE in context. When quadratic features are added, the feature space expands from 10 original features to 65 polynomial features.

Because the underlying relationship is inherently linear, the added features provide no additional predictive power. In conjunction with the training set fitting 65 parameters out of a total of 70 samples it leads to clearly illustrated overfitting due to high variance. As a result, as the model fits the training data closely, however it fails to accurately predict unseen test points as the model maps itself to the training data itself, not to any particular relationship.

And this is observed in the numerical output of the MSE as when additional polynomial features are added, the training set MSE drops from ~0.232 to ~0.010. On the other hand the test set MSE jumps from 0.272 to 4.290.